In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import os
from pathlib import Path
from pprint import pprint
import sys
from typing import Optional, List, Dict, Any, Tuple
if '..' not in sys.path: sys.path.append('..')

from dataclasses import dataclass
from datasets import load_dataset
from datasets.arrow_dataset import Dataset
import numpy as np
from matplotlib import pyplot as plt
from pydantic_yaml import parse_yaml_file_as
import torch
from torch import nn
import torch.nn.functional as F
from transformers import AutoTokenizer, PreTrainedTokenizer

from mllm.data.wiki.dswiki import WikiDsLoader
from mllm.data.utils import RandomInputTokenizer, RandomInputTokenizerV2, TokensSubset, TokensSubsetV2, tokens_subsets_to_tensors, tokens_subsets_v2_to_tensors
from mllm.exp.args import MIXED_DECODER_MODEL_CFG_FNAME
from mllm.model.mixed_decoder import MixedDecoder
from mllm.config.model import MixedDecoderCfg
from mllm.train.encdec_graph_bert import MaskedCiteDataset, create_masked_cite_dataloader, load_split_wiki_dataset

In [3]:
# DATA_PATH = Path(os.path.expandvars('$HOME')) / 'data'
DATA_PATH = Path('Q:/data')
# WIKI_DS_NAME = '20200501.en'
WIKI_DS_NAME = '20220301.en'

TRAIN_ENCDEC_BERT_PATH = DATA_PATH / 'train_mllm_encdec_bert'
mixed_decoder_subdir = 'mixeddecoder-20260304_105309-pre_encdecbert20260110193915-bertbaseuncased-d768-embEncCls-inp128-decGpt2-decmgpt2-msl384-sepT-pallF-eer4-ewn10x10-frzencF-trn_lr5e-05_bs30'

mixed_decoder_train_path = TRAIN_ENCDEC_BERT_PATH / mixed_decoder_subdir
mixed_decoder_snapshot_fpath = mixed_decoder_train_path / 'best.pth'
mixed_decoder_model_cfg_fpath = mixed_decoder_train_path / MIXED_DECODER_MODEL_CFG_FNAME

device_name = 'cpu'
# device_name = 'cuda'

device = torch.device(device_name)
print(device)

cpu


## Wikipedia dataset loading

In [4]:
# dss = load_dataset('wikipedia', WIKI_DS_NAME, beam_runner='DirectRunner', cache_dir=str(DATA_PATH))
dss = load_dataset('wikipedia', WIKI_DS_NAME, cache_dir=str(DATA_PATH), trust_remote_code=True)
ds: Dataset = dss['train']
n_docs = len(ds)
print(f'Wikipedia {WIKI_DS_NAME} docs: {n_docs}')

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

Wikipedia 20220301.en docs: 6458670


### Masked dataset loading

In [47]:
model_cfg = parse_yaml_file_as(MixedDecoderCfg, mixed_decoder_model_cfg_fpath)
pprint(model_cfg.dict())

tkz = AutoTokenizer.from_pretrained(model_cfg.enc_bert.pretrained_model_name)
inp_len = model_cfg.enc_bert.inp_len
n_special_toks = 3
prompt_all = model_cfg.prompt_all
mask_cfg = None
ds_cite = MaskedCiteDataset(ds, tkz, max_seq_len=inp_len, n_special_toks=n_special_toks, mask_cfg=mask_cfg, prompt_all=prompt_all, device=device)

{'d_model': 768,
 'decoder_model_name': 'gpt2',
 'decoder_type': <MixedDecoderType.Gpt2: 'gpt2'>,
 'emb_exp_rate': 4,
 'emb_win_max_size': 10,
 'emb_win_min_size': 10,
 'enc_bert': {'d_model': 768,
              'emb2_tok_name': '',
              'emb_type': <BertEmbType.Cls: 'cls'>,
              'inp_len': 128,
              'pad_token_id': 0,
              'pretrained_model_name': 'bert-base-uncased',
              'tokenizer_name': 'bert-base-uncased'},
 'max_seq_len': 384,
 'prompt_all': False,
 'train_cfg': {'batch_size': 30,
               'freeze_encoder': False,
               'learning_rate': 5e-05,
               'learning_rate_scheduler': {'cls_name': 'ReduceLROnPlateau',
                                           'module_path': 'torch.optim.lr_scheduler',
                                           'params': {'factor': 0.5,
                                                      'min_lr': 1e-08,
                                                      'mode': 'min',
            

### Inspect a masked dataset batch samples

In [30]:
batch_off = 10
batch_size = 5
inds = np.arange(batch_off, batch_off + batch_size)
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

In [31]:
print(tkz)

BertTokenizerFast(name_or_path='bert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)


In [32]:
print(f'inp_toks: {batch.inp_toks.shape}. inp_attn_mask: {batch.inp_att_mask.shape}. first: {batch.inp_toks[0, 0]}. last: {batch.inp_toks[0, -1]}.')
print(f'prompts_toks: {batch.prompts_toks.shape}. prompts_attn_mask: {batch.prompts_att_mask.shape}. first: {batch.prompts_toks[0, 0]}. last: {batch.prompts_toks[0, -1]}.')
print(f'cites_toks: {batch.cites_toks.shape}. cites_attn_mask: {batch.cites_att_mask.shape}.')

inp_toks: torch.Size([5, 128]). inp_attn_mask: torch.Size([5, 128]). first: 101. last: 102.
prompts_toks: torch.Size([5, 29]). prompts_attn_mask: torch.Size([5, 29]). first: 101. last: 102.
cites_toks: torch.Size([5, 84]). cites_attn_mask: torch.Size([5, 84]).


In [33]:
print('inp_toks[0]:', tkz.decode(batch.inp_toks[0].cpu().numpy()))
print('prompts_toks[0]:', tkz.decode(batch.prompts_toks[0].cpu().numpy()))
print('cites_toks[0]:', tkz.decode(batch.cites_toks[0].cpu().numpy()))

inp_toks[0]: [CLS] at the clusters fluttered gujarat initial ceremonies were gold - plated solid bronze. within a few years, the bronze was abandoned in favor of britannia metal, a pewter - like alloy which is then plated in copper, nickel silver, and finally, 24 - karat gold. due to a metal shortage during world war ii, oscars were made of painted plaster for three years. following the war, the academy invited recipients to red 32 olympiad buckeem the plaster figures for gold - plated metal ones. the only addition to the oscar since it was created is a minor streamlining of the base. the original oscar mold was [SEP]
prompts_toks[0]: [CLS] cite tag begin : " clusters fluttered gujarat ". cite tag end : " 32 olympiad buck ". produce output text between these tags. [SEP]
cites_toks[0]: initial ceremonies were gold - plated solid bronze. within a few years, the bronze was abandoned in favor of britannia metal, a pewter - like alloy which is then plated in copper, nickel silver, and final

## Model loading and inference

In [48]:
chkpt = torch.load(mixed_decoder_snapshot_fpath, map_location=device)
model = MixedDecoder(model_cfg, tkz).to(device)
model.load_pretrained(chkpt)
del chkpt
model.eval()
None

Load 351


### Run forward pass (with windowing disabled for inference)

In [69]:
# Temporarily disable windowing for deterministic inference
orig_emb_win_max_size = model.cfg.emb_win_max_size
model.cfg = model.cfg.copy(update={'emb_win_max_size': 0})

batch_off = 10
batch_size = 10
inds = np.arange(batch_size) + batch_off
inds = inds.tolist()
batch = ds_cite.get_batch(inds)

In [70]:
with torch.no_grad():
    loss_dict, dec_logits = model.run_on_text_citation(batch)
print(loss_dict)

{'loss': tensor(2.2775)}


In [53]:
logits = dec_logits.detach().numpy()
logits.shape

(5, 167, 50257)

In [54]:
probs_pred = torch.softmax(dec_logits, dim=-1)
print('probs shape:', probs_pred.shape)
docs_toks_out = torch.argmax(probs_pred, dim=-1)
docs_toks_out = docs_toks_out.detach().numpy()
print('toks_out shape:', docs_toks_out.shape)

probs shape: torch.Size([5, 167, 50257])
toks_out shape: (5, 167)


In [55]:
# Compute target_start_idx to extract only the target region from logits
# Layout: [ctx_embs(n_ctx), (sep), prompt_toks, target_toks_input(T-1)]
# Target labels start at target_start_idx = prefix_len - 1 (last prompt position predicts first target)
n_ctx = batch_size  # all batch CLS embeddings used as context
if model.cfg.emb_exp_rate > 0:
    n_ctx = batch_size * model.cfg.emb_exp_rate
prefix_len = n_ctx
if model.cfg.use_sep:
    prefix_len += 1
prefix_len += batch.prompts_toks.shape[1]
target_start_idx = prefix_len - 1

if model.cfg.prompt_all:
    target_len = batch.inp_toks.shape[1] - 1  # strip CLS
else:
    target_len = batch.cites_toks.shape[1]

print(f'n_ctx: {n_ctx}, prefix_len: {prefix_len}, target_start_idx: {target_start_idx}, target_len: {target_len}')

# Extract predicted tokens for the target region
target_logits = dec_logits[:, target_start_idx:target_start_idx + target_len, :]
target_toks_pred = torch.argmax(target_logits, dim=-1).detach().numpy()
print('target_toks_pred shape:', target_toks_pred.shape)

n_ctx: 20, prefix_len: 50, target_start_idx: 49, target_len: 118
target_toks_pred shape: (5, 118)


In [56]:
for i in range(batch_size):
    print(f'=== Sample {i} ===')
    if model.cfg.prompt_all:
        gt_toks = batch.inp_toks[i, 1:]  # strip CLS
    else:
        gt_toks = batch.cites_toks[i]
    print('ground truth:')
    print(tkz.decode(gt_toks.cpu().numpy()))
    print('predicted:')
    print(tkz.decode(target_toks_pred[i]))
    print()

=== Sample 0 ===
ground truth:
spread their message. anarchists have found it easier to create websites because of distributional and other difficulties, hosting electronic libraries and other portals. anarchists were also involved in developing various software that are available for free. the way these hacktivists work to develop and distribute resembles the anarchist ideals, especially when it comes to preserving users ' privacy from state surveillance. anarchists organize [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
predicted:
and, in, to to. in, to,, to to,. 2,,, to and the and, the educational. its, also used in the the organizations, have often to the and the anarchist of organizationsistss and, create and create the, activitiess. and in they is to the the. rights. the.

## Encoder embeddings inspection

In [57]:
with torch.no_grad():
    inp_enc_embs = model.run_enc(batch.inp_toks, batch.inp_att_mask)
print('inp_enc_embs shape:', inp_enc_embs.shape)

inp_embs = inp_enc_embs.cpu().detach().numpy()
print('min:', inp_embs.min(), 'max:', inp_embs.max(), 'mean:', inp_embs.mean(), 'std:', inp_embs.std())

inp_enc_embs shape: torch.Size([5, 768])
min: -4.2856402 max: 3.401846 mean: -0.021137591 std: 1.0188017


## Autoregressive generation from encoder embeddings

In [58]:
@torch.no_grad()
def generate_from_embs(
    model: MixedDecoder, inp_enc_embs: torch.Tensor,
    prompt_toks: torch.Tensor, prompt_att_mask: torch.Tensor,
    max_new_tokens: int = 100, temperature: float = 1.0,
) -> torch.Tensor:
    """Autoregressive generation: given encoder CLS embeddings and prompt tokens,
    generate tokens one by one using the MixedDecoder.

    Args:
        model: MixedDecoder model in eval mode
        inp_enc_embs: (batch_size, d_model) - CLS embeddings from encoder
        prompt_toks: (batch_size, prompt_len) - prompt token ids
        prompt_att_mask: (batch_size, prompt_len) - prompt attention mask
        max_new_tokens: maximum number of tokens to generate
        temperature: sampling temperature (1.0 = greedy with argmax)

    Returns:
        generated_toks: (batch_size, max_new_tokens) - generated token ids
    """
    batch_size = inp_enc_embs.shape[0]
    device = inp_enc_embs.device
    cfg = model.cfg

    # Build context embeddings (no windowing for generation)
    if cfg.emb_exp_rate > 0:
        exp_embs = model.emb_exp(inp_enc_embs)
        exp_embs = exp_embs.view(batch_size, cfg.emb_exp_rate, model.d_dec)
        exp_embs = exp_embs.reshape(1, batch_size * cfg.emb_exp_rate, model.d_dec)
        ctx_embs = exp_embs.expand(batch_size, -1, -1)
    else:
        ctx_embs = inp_enc_embs.unsqueeze(0).expand(batch_size, -1, -1)

    # Project if needed
    if model.enc_proj is not None:
        ctx_embs = model.enc_proj(ctx_embs)

    n_ctx = ctx_embs.shape[1]

    # Build initial prefix: [ctx_embs, (sep), prompt_embs]
    parts_embs = [ctx_embs]
    parts_mask = [torch.ones((batch_size, n_ctx), dtype=torch.long, device=device)]

    if cfg.use_sep:
        sep_tok = torch.full((batch_size, 1), model.sep_token_id, dtype=torch.long, device=device)
        sep_emb = model.word_embeddings(sep_tok)
        parts_embs.append(sep_emb)
        parts_mask.append(torch.ones((batch_size, 1), dtype=torch.long, device=device))

    prompt_embs = model.word_embeddings(prompt_toks)
    parts_embs.append(prompt_embs)
    parts_mask.append(prompt_att_mask)

    input_embs = torch.cat(parts_embs, dim=1)  # (batch_size, prefix_len, d_dec)
    attention_mask = torch.cat(parts_mask, dim=1)

    generated_ids = []
    for step in range(max_new_tokens):
        total_len = input_embs.shape[1]
        pos_ids = torch.arange(total_len, device=device).unsqueeze(0)
        pos_embs = model.pos_emb(pos_ids)
        embs_with_pos = input_embs + pos_embs

        logits = model.run_decoder(embs_with_pos, attention_mask)
        next_logits = logits[:, -1, :]  # (batch_size, vocab_size)

        if temperature <= 0:
            next_tok = torch.argmax(next_logits, dim=-1)  # (batch_size,)
        else:
            probs = torch.softmax(next_logits / temperature, dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1).squeeze(-1)

        generated_ids.append(next_tok)

        # Append new token embedding
        next_emb = model.word_embeddings(next_tok.unsqueeze(1))  # (batch_size, 1, d_dec)
        input_embs = torch.cat([input_embs, next_emb], dim=1)
        attention_mask = torch.cat([attention_mask, torch.ones((batch_size, 1), dtype=torch.long, device=device)], dim=1)

    return torch.stack(generated_ids, dim=1)  # (batch_size, max_new_tokens)

In [59]:
max_new_tokens = 100
gen_toks = generate_from_embs(
    model, inp_enc_embs,
    batch.prompts_toks, batch.prompts_att_mask,
    max_new_tokens=max_new_tokens, temperature=0,
)
print('gen_toks shape:', gen_toks.shape)

for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input:')
    print(tkz.decode(batch.inp_toks[i].cpu().numpy()))
    print('prompt:')
    print(tkz.decode(batch.prompts_toks[i].cpu().numpy()))
    print('generated:')
    print(tkz.decode(gen_toks[i].cpu().numpy()))
    print()

gen_toks shape: torch.Size([5, 100])
=== Sample 0 ===
input:
[CLS] a syndicate. as in the past, newspapers and journals are used, and anarchists have gone online in the world wide web tozierection canadians spread their message. anarchists have found it easier to create websites because of distributional and other difficulties, hosting electronic libraries and other portals. anarchists were also involved in developing various software that are available for free. the way these hacktivists work to develop and distribute resembles the anarchist ideals, especially when it comes to preserving users ' privacy from state surveillance. anarchists organize nebraskaova children themselves to squat and reclaim public spaces. during important events such as protests and when spaces are being [SEP]
prompt:
[CLS] cite tag begin : "zierection canadians ". cite tag end : " nebraskaova children ". produce output text between these tags. [SEP]
generated:
and,. to to to.... in...,. 2.,.. their slogans h

## Decoding with noisy embeddings

In [ ]:
# Add small noise to encoder embeddings and compare generation quality
noise_std = 1e-2
noise = torch.randn_like(inp_enc_embs) * noise_std
inp_enc_embs_noisy = inp_enc_embs + noise

gen_toks_noisy = generate_from_embs(
    model, inp_enc_embs_noisy,
    batch.prompts_toks, batch.prompts_att_mask,
    max_new_tokens=max_new_tokens, temperature=0,
)

for i in range(batch_size):
    print(f'=== Sample {i} ===')
    print('input:')
    print(tkz.decode(batch.inp_toks[i].cpu().numpy()))
    print('generated (clean):')
    print(tkz.decode(gen_toks[i].cpu().numpy()))
    print('generated (noisy, std={:.0e}):'.format(noise_std))
    print(tkz.decode(gen_toks_noisy[i].cpu().numpy()))
    print()

## Restore original config

In [ ]:
# Restore original emb_win_max_size
model.cfg = model.cfg.copy(update={'emb_win_max_size': orig_emb_win_max_size})
print('emb_win_max_size restored to:', model.cfg.emb_win_max_size)